# Libraries

In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split



# Load Dataset

In [2]:

df = pd.read_csv("./data/dataset_lstm.csv")
df.head()

,Unnamed: 0,R_-239,R_-238,R_-237,R_-236,R_-235,R_-234,R_-233,R_-232,R_-231,...,R_-6,R_-5,R_-4,R_-3,R_-2,R_-1,R_0,ticker,date,target
0,0,-0.768984,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,...,1.106459,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,A,2014-01-23,0
1,1,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,...,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,A,2014-01-24,1
2,2,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,...,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,A,2014-01-27,0
3,3,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,...,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,A,2014-01-28,1
4,4,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,-0.442705,...,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,-0.565664,A,2014-01-29,1


# Split dataset into training and validation

In [3]:
def normalize_and_split_study_period(df, train_days=750):
    df_seq = df
    
    # 1. Encontrar los días únicos en este bloque exacto
    unique_dates = sorted(df_seq['date'].unique())
    
    # Si por alguna razón el bloque tiene menos de 750 días, ajustamos el índice
    split_idx = min(train_days - 1, len(unique_dates) - 1)
    split_date = unique_dates[split_idx] 
    
    # 2. Separar Train y Test estrictamente por cronología
    train_df = df_seq[df_seq['date'] <= split_date].copy()
    test_df = df_seq[df_seq['date'] > split_date].copy()
    
    # 3. Obtener solo las columnas matemáticas (R_-239 a R_0)
    return_cols = [col for col in df_seq.columns if col.startswith('R_')]

    return train_df[return_cols+["target"]], test_df[return_cols+["target"]]
    
return_cols = [col for col in df.columns if col.startswith('R_')]

train_df, test_df = normalize_and_split_study_period(df)


X_train = train_df[return_cols]
y_train = train_df["target"]

In [4]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [5]:
train_df.shape

(311200, 241)

In [6]:
test_df.shape

(86939, 241)

In [7]:
train_df.head()

,R_-239,R_-238,R_-237,R_-236,R_-235,R_-234,R_-233,R_-232,R_-231,R_-230,...,R_-8,R_-7,R_-6,R_-5,R_-4,R_-3,R_-2,R_-1,R_0,target
0,-0.768984,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,...,0.600742,-0.022987,1.106459,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,0
1,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,...,-0.022987,1.106459,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,1
2,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,...,1.106459,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,0
3,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,-0.442705,...,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,1
4,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,-0.442705,0.737066,...,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,-0.565664,1


# Train model with LSTM

In [11]:
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPU:", tf.config.list_physical_devices('GPU'))

Built with CUDA: False
GPU: []


In [12]:
import tensorflow as tf
print(tf.__version__)

2.19.1


In [8]:

model = Sequential()

# Capa LSTM
model.add(
    LSTM(
        units=25,
        input_shape=(240, 1),
        dropout=0.1,
        recurrent_dropout=0.1
    )
)

# Capa de salida
model.add(
    Dense(
        units=2,
        activation='softmax'
    )
)

# COMPILACION
optimizer = RMSprop()

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# EARLY STOPPING

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


# ENTRENAMIENTO

history = model.fit(
    X_tr,
    y_tr,
    epochs=1000,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    shuffle=True
)

# =========================================================
# RESUMEN DEL MODELO
# =========================================================

model.summary()

D:\ADMIN\Documents\NN\NeuralNetworks-Group1\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/1000
4139/7780 ━━━━━━━━━━━━━━━━━━━━ 4:54 81ms/step - accuracy: 0.5047 - loss: 0.6935

KeyboardInterrupt: 